# Kommentarer på vad vi ska göra

* Vi kan ha en champion för varje och säga att vi hitta genom att göra CV samt med grid search, man kan nämnn generellt vad man testat

* Komma på vilken metric är vettig att dra resultat av och ge till user

* Ha några plots, som train/val loss, confusion matrix och vad mer vi tycker är intressant

* Ha animering på video på cut video?? 

* Om tid finns kanske lägga till conv NN

* Göra allt för Kinect och media pipe data, ha olika champions då borde vara bäst?? Jämföra med varandra


# Sebastians generella instruktioner från moodle 

* Submit a single, self-contained, and fully runnable notebook - I do run and debug all of your notebooks. While I do inspect your repositories, you must not rely on me trying to find where the training code is, it needs to be in the notebook.

* Do not include the datasets or pre-rendered plots. Do not put Python in formatted markdown cells. Once I run the notebook, it computes all results. If your notebook relies on earlier results or produced data, you should include it though.

* Of course you can keep the reporting for all trained variants and champion models, etc. But the notebook needs to contain one runnable version of each variant so to speak.

* Submit just the notebook, not a Zip. Only a Zip if your notebook requires auxiliary files to run.

## Handle all inputs

In [2]:
import torch.nn as nn
import torch
import mlflow
import random
import os
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from datetime import datetime
from sklearn.model_selection import KFold
import torch.optim as optim
import json
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Setup

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

# make sure everything has same seeds
random_seed = 42
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Squat_or_not")
mlflow.enable_system_metrics_logging()

MlflowException: API request to endpoint /api/2.0/mlflow/experiments/get-by-name failed with error code 403 != 200. Response body: ''

## Load data 

* First load from chosen folder 
* Split into X and Y ignore frame NO feature
* Do normalisation with base in train data on all partitions




## Create the new Y value using the kinetic cut as reference

# **Feed forward network**


## Functions for the implementation

In [ ]:
# Function to load champion model/ best performing model
def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None

# Function for saving a model as our current champion model
def save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "f1": float(f1),
        "recall": float(recall),
        "precision": float(precision),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# Function that saves a model as champion uisng the save_champion_model function if either 1. We have no champion, 2. New model is better
def update_champion(metadata_dir, champion_dir, model, model_name, f1, recall, precision, hyperparameters):
    current = load_champion_info(metadata_dir)

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters)

    elif f1 > current["f1"]:
        print(f"New model is better (f1 {f1} > {current['f1']})")
        save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters)

    else:
        print(f"Model NOT better (f1 {f1} < {current['f1']})")


# Define initial weights and biases
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight) # good for ReLU
        nn.init.zeros_(m.bias)


# Define Feedforward squat classifier class
class Squatclassifier(nn.Module):
    def __init__(self, input_size, hidden_layers: list, activation="relu", dropout=0.0):
        super().__init__()

        layers = []

        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        act = activations[activation]
        prev_size = input_size

        # Hidden layers
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(act)

            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_size = hidden_size

        # Output layer- 1 neuron
        layers.append(nn.Linear(prev_size, 1))

        self.network = nn.Sequential(*layers)
        self.network.apply(init_weights)
    
    def forward(self, X):
        return self.network(X)
        


# Initializes a squatclassifier model based on a configuration
def build_model(config):
    return Squatclassifier(
        input_size=config["input_size"],
        hidden_layers=config["layers"],
        activation=config["activation"],
        dropout=config["dropout"]
    ).to(device)


# Train the model
def train_one_model(model, config, x_train, y_train, x_val, y_val, loss_fn):
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    epochs = config["epochs"]

    best_val_f1 = 0
    best_state = None
    patience = 10
    epochs_no_improve = 0

    # reshape targets
    y_train = y_train.view(-1, 1)
    y_val = y_val.view(-1, 1)

    # Training loop (full-batch)
    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        logits = model(x_train)
        loss = loss_fn(logits, y_train)

        loss.backward()
        optimizer.step()

        # Validation
        model.eval()

        with torch.no_grad():
            val_logits = model(x_val)
            probs = torch.sigmoid(val_logits)
            preds = (probs > 0.5).float()

            val_f1 = f1_score(
                y_val.cpu().numpy(),
                preds.cpu().numpy()
            )

        # Early stopping on f1-score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    return best_val_f1, best_state


# Cross validation function
def cross_validation(config, X, Y, k):

    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    fold_scores = []
    fold_models = []

    # Train and evaluate on each fold
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        x_train = X[train_idx].to(device)
        y_train = Y[train_idx].to(device)

        x_val = X[val_idx].to(device)
        y_val = Y[val_idx].to(device)

        # Build model
        model = build_model(config)


        val_f1, best_state = train_one_model(
            model,
            config,
            x_train,
            y_train,
            x_val,
            y_val,
            # Binary Cross Entropy/Log Loss
            loss_fn=nn.BCEWithLogitsLoss()
        )

        fold_scores.append(val_f1)
        fold_models.append(best_state)


    avg_f1 = sum(fold_scores) / len(fold_scores)


    return {
        "cv_mean_f1": avg_f1,
        "fold_scores": fold_scores,
        "fold_models": fold_models
    }

## Setup

In [ ]:
# Gridsearch and cross validation where we decided upon the hyperparameter is in HugoProject/notebooks/assignment11_hugo.ipynb

# Access where best FNN classificator is stored
model_root = "binary_classificator_models/fnn_classificator"
metadata_path = os.path.join(model_root, "metadata", "champion_info.json")


champion_info = load_champion_info(metadata_path)
best_config = champion_info["hyperparameters"]
print("Champion config loaded")
print(best_config)


# Alternatively here is the config stored in local variable
# best_config = {"model_name": "final_model",
#                "saved_at": "2026-04-28 20:11:13",
#                "f1": 0.9239302694136292,
#                "recall": 0.938430583501006,
#                "precision": 0.9098712446351931,
#                "hyperparameters": {
#                 "layers": [
#                   256,
#                   128,
#                   32
#                 ],
#                 "lr": 0.005,
#                 "dropout": 0.1,
#                 "activation": "relu",
#                 "epochs": 200,
#                 "input_size": 39
#                 }
# }

## Re-train

In [ ]:
# Rebuild best model from cross validation grid search
final_model = build_model(best_config)

# Make split for the retraining s.t model isn't overfitted to the training data
x_tr, x_val, y_tr, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42)

# Retrain best config from the cross validation gridsearch
_, best_state = train_one_model(final_model, best_config, x_tr, y_tr, x_val, y_val, loss_fn = nn.BCEWithLogitsLoss())


## Evaluate and results



In [ ]:
# Load weights from re-training
final_model.load_state_dict(best_state)

# Evaluate against test set
final_model.eval()
with torch.no_grad():
    logits = final_model(x_test.to(device))
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    f1 = f1_score(y_test.cpu(), preds.cpu())
    precision = precision_score(y_test.cpu(), preds.cpu())
    recall = recall_score(y_test.cpu(), preds.cpu())
    accuracy = accuracy_score(y_test.cpu(), preds.cpu())

    cm = confusion_matrix(y_test.cpu(), preds.cpu())

print("TEST RESULTS")
print(f"F1: {f1}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"Accuracy: {round(accuracy * 100, 4)}%")
print("\nConfusion Matrix for best performing FNN:")

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Squat", "Squat"],
            yticklabels=["Not Squat", "Squat"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix for best performing FNN")
plt.show()

# Reccurant NN

## Setup

CV mean F1: 0.9480  |  CV Std F1: 0.014749553665563356

per fold: [0.9574, 0.9629, 0.9428, 0.9282, 0.9703, 0.9432, 0.9576, 0.9521, 0.9244, 0.9415]


{
  "model_name": "lstm_cv",
  "saved_at": "2026-04-28 22:58:29",
  "f1": 0.9375,
  "recall": 0.9064612057312788,
  "precision": 0.9303551609322974,
  "hyperparameters": {
    "hidden_layers": [
      160,
      160,
      64
    ],
    "layer_type": "LSTM",
    "dropout": 0.4,
    "learning_rate": 0.0005,
    "batch_size": 32,
    "seq_length": 30,
    "stride": 30,
    "weight_decay": 1e-06
  },
  "test_results": {
    "f1": 0.895628078817734,
    "accuracy": 0.8758241758241758,
    "precision": 0.8772617611580217,
    "recall": 0.9147798742138364,
    "auc": 0.9539372310493214,
    "evaluated_at": "2026-04-28 23:01:18"
  }
}


## Load Best model

In [ ]:
pre_trained_model = torch.load("binary_classificator_models/recurrant_classification/champion_model.pt") # trained on kinetic




## Train new model

In [ ]:
def compute_classification_metrics(logits, targets):
    """Convert logits -> binary preds and compute accuracy, F1, precision, recall."""
    probs  = torch.sigmoid(logits).cpu().numpy().flatten()
    preds  = (probs > 0.5).astype(int)
    labels = targets.cpu().numpy().flatten().astype(int)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "probs":     probs,
        "preds":     preds,
        "labels":    labels,
    }


with mlflow.start_run(run_name=params_lstm["run_name"]) as run:
    mlflow.log_params(params_lstm)

    train_dataset = TensorDataset(train_x, train_y)
    val_dataset   = TensorDataset(val_x,   val_y)
    test_dataset  = TensorDataset(test_x,  test_y)

    train_loader = DataLoader(train_dataset, batch_size=params_lstm["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=params_lstm["batch_size"], shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=params_lstm["batch_size"], shuffle=False)

    best_val_f1       = -1.0
    best_model_state  = None
    epochs_no_improve = 0

    for epoch in range(params_lstm["epochs"]):

        # ── Training ─────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        all_train_logits, all_train_labels = [], []

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss   = loss_fn(logits, batch_y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            all_train_logits.append(logits.detach())
            all_train_labels.append(batch_y)

        avg_train_loss   = np.mean(train_losses)
        train_logits_cat = torch.cat(all_train_logits)
        train_labels_cat = torch.cat(all_train_labels)
        train_m          = compute_classification_metrics(train_logits_cat, train_labels_cat)

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        all_val_logits, all_val_labels = [], []

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                logits  = model(batch_x)
                val_losses.append(loss_fn(logits, batch_y).item())
                all_val_logits.append(logits)
                all_val_labels.append(batch_y)

        avg_val_loss   = np.mean(val_losses)
        val_logits_cat = torch.cat(all_val_logits)
        val_labels_cat = torch.cat(all_val_labels)
        val_m          = compute_classification_metrics(val_logits_cat, val_labels_cat)

        mlflow.log_metrics({
            "train_loss":      avg_train_loss,
            "train_accuracy":  train_m["accuracy"],
            "train_f1":        train_m["f1"],
            "train_precision": train_m["precision"],
            "train_recall":    train_m["recall"],
            "val_loss":        avg_val_loss,
            "val_accuracy":    val_m["accuracy"],
            "val_f1":          val_m["f1"],
            "val_precision":   val_m["precision"],
            "val_recall":      val_m["recall"],
        }, step=epoch)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{params_lstm['epochs']} "
                  f"| train_loss: {avg_train_loss:.4f}  train_f1: {train_m['f1']:.4f} "
                  f"| val_loss: {avg_val_loss:.4f}  val_f1: {val_m['f1']:.4f}  "
                  f"val_acc: {val_m['accuracy']:.4f}")

        # ── Early stopping on val F1 ──────────────────────────────────────────
        if params_lstm.get("patience"):
            if val_m["f1"] > best_val_f1:
                best_val_f1      = val_m["f1"]
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
                print(f"  → New best val F1: {best_val_f1:.4f} (epoch {epoch+1})")
            else:
                epochs_no_improve += 1
            if epochs_no_improve >= params_lstm["patience"]:
                print(f"Early stopping at epoch {epoch+1} — best val F1: {best_val_f1:.4f}")
                break

    # ── Restore best model before test eval ──────────────────────────────────
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Restored best model (val F1: {best_val_f1:.4f})")
        
        # ========== SAVE CANDIDATE MODEL ==========
        # Save this candidate model for reference
        candidates_dir = "candidates"
        os.makedirs(candidates_dir, exist_ok=True)
        candidate_name = f"{params_lstm['run_name']}_val_f1_{best_val_f1:.4f}"
        save_candidate_model(model, candidate_name, candidates_dir)
        
        # ========== TRY TO UPDATE CHAMPION ==========
        metadata_dir = "champion_metadata"
        champion_dir = "champion"
        
        # Prepare hyperparameters for logging
        hyperparameters = {
            "hidden_layers": params_lstm["hidden_layers"],
            "layer_type": params_lstm["layer_type"],
            "dropout": params_lstm["dropout"],
            "learning_rate": params_lstm["learning_rate"],
            "batch_size": params_lstm["batch_size"],
            "seq_length": params_lstm["seq_length"],
            "stride": params_lstm["stride"],
            "weight_decay": params_lstm["weight_decay"],
        }
        
        # Check if this model beats the champion
        was_champion_updated = update_champion(
            metadata_dir=metadata_dir,
            champion_dir=champion_dir,
            model=model,
            model_name=params_lstm["run_name"],
            f1=best_val_f1,  # Using validation F1 for champion selection
            recall=val_m["recall"],
            precision=val_m["precision"],
            hyperparameters=hyperparameters
        )
        
        if was_champion_updated:
            print("🏆 This model is the NEW CHAMPION!")
        else:
            print("📁 Model saved as candidate (champion remains unchanged)")

    # ── Test evaluation ───────────────────────────────────────────────────────
    model.eval()
    test_losses = []
    all_test_logits, all_test_labels = [], []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits  = model(batch_x)
            test_losses.append(loss_fn(logits, batch_y).item())
            all_test_logits.append(logits)
            all_test_labels.append(batch_y)

    avg_test_loss   = np.mean(test_losses)
    test_logits_cat = torch.cat(all_test_logits)
    test_labels_cat = torch.cat(all_test_labels)
    test_m          = compute_classification_metrics(test_logits_cat, test_labels_cat)

    test_auc = roc_auc_score(test_m["labels"], test_m["probs"])

    mlflow.log_metrics({
        "test_loss":      avg_test_loss,
        "test_accuracy":  test_m["accuracy"],
        "test_f1":        test_m["f1"],
        "test_precision": test_m["precision"],
        "test_recall":    test_m["recall"],
        "test_auc":       test_auc,
    })

    mlflow.pytorch.log_model(model, artifact_path=f"best_model_{params_lstm['run_name']}")

    # After computing test results, update champion metadata
    def update_champion_with_test_results(metadata_dir, test_m, test_auc):
        """Add test results to champion info after evaluation"""
        info_path = os.path.join(metadata_dir, "champion_info.json")
        
        if os.path.exists(info_path):
            with open(info_path, "r") as f:
                info = json.load(f)
            
            # Add test results
            info["test_results"] = {
                "f1": float(test_m["f1"]),
                "accuracy": float(test_m["accuracy"]),
                "precision": float(test_m["precision"]),
                "recall": float(test_m["recall"]),
                "auc": float(test_auc),
                "evaluated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
            
            with open(info_path, "w") as f:
                json.dump(info, f, indent=2)
            print("✅ Champion info updated with test results")
        else:
            print("⚠️ No champion info found to update")

    update_champion_with_test_results("champion_metadata", test_m, test_auc)

    print(f"\n✅ Test Results:")
    print(classification_report(test_m["labels"], test_m["preds"],
                                target_names=["Not movement", "Movement"]))

    # ── Confusion matrix + ROC curve ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    cm = confusion_matrix(test_m["labels"], test_m["preds"])
    ConfusionMatrixDisplay(cm, display_labels=["Not movement", "Movement"]).plot(
        ax=axes[0], colorbar=False
    )
    axes[0].set_title("Confusion Matrix (Test set)")

    RocCurveDisplay.from_predictions(
        test_m["labels"], test_m["probs"],
        name=f"(AUC={test_auc:.3f})",
        ax=axes[1]
    )
    axes[1].set_title("ROC Curve (Test set)")

    plt.tight_layout()
    plt.savefig("test_evaluation.png", dpi=150)
    mlflow.log_artifact("test_evaluation.png")
    plt.show()

    # ── Temporal prediction plot ──────────────────────────────────────────────
    n_frames_to_plot = params_lstm["seq_length"] * 10  # Reduced to 10 for better visibility
    first_seq_true  = test_m["labels"][:n_frames_to_plot]
    first_seq_probs = test_m["probs"][:n_frames_to_plot]

    fig2, ax = plt.subplots(figsize=(12, 3))
    ax.plot(first_seq_true,  label="True label",      linewidth=2)
    ax.plot(first_seq_probs, label="Predicted prob",  linewidth=2, linestyle="--")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Movement label")
    ax.set_title("Frame-level predictions over time (first test sequence)")
    ax.legend()
    ax.set_ylim(-0.1, 1.1)
    plt.tight_layout()
    plt.savefig("temporal_predictions.png", dpi=150)
    mlflow.log_artifact("temporal_predictions.png")
    plt.show()

    print(f"\nAUC: {test_auc:.4f}  |  F1: {test_m['f1']:.4f}  |  Accuracy: {test_m['accuracy']:.4f}")

## Evaluate and results


# Cut video 

# Summary and future works